# Bottle Vision — SAM 3 Colab Runner

Это рабочая схема из исходной копии: Colab использует системный Python только для установки `uv`, а SAM 3 и Bottle Vision запускаются в отдельном `.venv312` на Python 3.12.

Для Tesla T4 оставляем этот runtime без упрощений: PyTorch 2.10.0 + TorchVision 0.25.0 + CUDA 12.8. Dtype-fix находится в `src/bottle_vision/segmentation/sam3.py`, а не в случайном monkey-patch ноутбука.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO = Path('/content/Bottlevision')
REMOTE = 'https://github.com/Rollerboy22/Bottlevision.git'

if REPO.exists():
    subprocess.run(['rm', '-rf', str(REPO)], check=True)

subprocess.run(['git', 'clone', '--branch', 'main', '--depth', '1', REMOTE, str(REPO)], check=True)
%cd /content/Bottlevision
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('Colab Python:', sys.version.split()[0])


In [ ]:
import subprocess, sys
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'], check=True)
UV = subprocess.check_output(['bash', '-lc', 'command -v uv || echo /root/.local/bin/uv'], text=True).strip()

VENV = REPO / '.venv312'
subprocess.run([UV, 'python', 'install', '3.12'], check=True)
subprocess.run([UV, 'venv', '--python', '3.12', str(VENV), '--clear'], check=True)

PYTHON = VENV / 'bin' / 'python'
print('ML Python:', subprocess.check_output([str(PYTHON), '-c', 'import sys; print(sys.version)'], text=True).strip())


In [ ]:
import subprocess

UV = subprocess.check_output(['bash', '-lc', 'command -v uv || echo /root/.local/bin/uv'], text=True).strip()
PYTHON = str(REPO / '.venv312' / 'bin' / 'python')

# Тот же ML runtime, что был в рабочей копии ноутбука.
subprocess.run([UV, 'pip', 'install', '--python', PYTHON,
    'torch==2.10.0', 'torchvision==0.25.0',
    '--index-url', 'https://download.pytorch.org/whl/cu128'
], check=True)

subprocess.run([UV, 'pip', 'install', '--python', PYTHON,
    'numpy==1.26.4',
    'pydantic>=2.7,<3', 'PyYAML>=6.0,<7', 'Pillow>=10,<12',
    'timm>=1.0.17', 'tqdm', 'ftfy==6.1.1', 'regex',
    'iopath>=0.1.10', 'typing_extensions', 'huggingface_hub>=0.30',
    'einops>=0.8', 'pycocotools', 'psutil',
    'opencv-python-headless', 'matplotlib', 'scikit-image',
    'git+https://github.com/facebookresearch/sam3.git'
], check=True)

print('Installation finished.')


In [ ]:
import subprocess

PYTHON = str(REPO / '.venv312' / 'bin' / 'python')
SRC = str(REPO / 'src')

check = f'''
import sys
sys.path.insert(0, {SRC!r})
import numpy, torch, torchvision, sam3, bottle_vision
print('Python:', sys.version.split()[0])
print('NumPy:', numpy.__version__)
print('PyTorch:', torch.__version__)
print('TorchVision:', torchvision.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Capability:', torch.cuda.get_device_capability(0))
print('SAM3: OK')
print('bottle_vision: OK')
'''
result = subprocess.run([PYTHON, '-c', check], capture_output=True, text=True)
print(result.stdout)
if result.stderr: print('STDERR:', result.stderr)
if result.returncode != 0: raise RuntimeError('Runtime verification failed')


## Hugging Face

Не хранить токен в ноутбуке. Если checkpoint требует авторизации, раскомментируй `login()` и введи токен в форме Colab.

In [ ]:
from huggingface_hub import login

# login()  # токен вводится интерактивно; не записывай его в notebook


In [ ]:
import subprocess
from pathlib import Path
from google.colab import files

REPO = Path('/content/Bottlevision')
PYTHON = str(REPO / '.venv312' / 'bin' / 'python')
SRC = str(REPO / 'src')

uploaded = files.upload()
if not uploaded: raise RuntimeError('Картинка не загружена')
filename, data = next(iter(uploaded.items()))
dst_path = REPO / 'colab_input.jpg'
dst_path.write_bytes(data)
print('Получен файл:', filename, 'размер:', len(data), 'байт')

run_script = REPO / 'colab' / '_run_once.py'
run_script.write_text(f'''\
import sys
sys.path.insert(0, {SRC!r})
import json
from pathlib import Path
import numpy as np
from PIL import Image
import torch
from bottle_vision.config import load_config
from bottle_vision.pipeline import run_pipeline
from bottle_vision.segmentation import make_review_views

image_path = Path({str(dst_path)!r})
output_dir = Path('/content/Bottlevision/colab_output')
output_dir.mkdir(exist_ok=True)
config = load_config(Path('/content/Bottlevision/configs/default.yaml'))
image = np.asarray(Image.open(image_path).convert('RGB'), dtype=np.uint8)
print('Shape:', image.shape)
print('GPU:', torch.cuda.get_device_name(0))
print('Capability:', torch.cuda.get_device_capability(0))
print('Запускаю pipeline...')
with torch.autocast(device_type='cuda', enabled=False):
    result = run_pipeline(image, config)
seg = result.segmentation
print('Модель:', seg.model_name)
print('Инстансов:', len(seg.instances))
if seg.error: print('Ошибка:', seg.error)
summary = {{
    'model': seg.model_name,
    'instance_count': len(seg.instances),
    'error': seg.error,
    'instances': [{{
        'id': inst.instance_id,
        'confidence': float(inst.confidence) if inst.confidence is not None else None,
        'accepted': bool(inst.metadata.get('accepted_by_gate', False)),
    }} for inst in seg.instances]
}}
(output_dir / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))
views = make_review_views(image, seg, alpha=0.45)
for name, arr in views.items():
    safe = name.lower().replace(' ', '_')
    Image.fromarray(np.asarray(arr, dtype=np.uint8)).save(output_dir / f'{{safe}}.png')
    print('Сохранено:', safe + '.png')
print('Готово')
''', encoding='utf-8')

result = subprocess.run([PYTHON, str(run_script)], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR (последние 3000 символов):')
    print(result.stderr[-3000:])
if result.returncode != 0:
    raise RuntimeError('Pipeline завершился с ошибкой; см. stderr выше.')


In [ ]:
from pathlib import Path
from IPython.display import display, Image as DisplayImage

out = Path('/content/Bottlevision/colab_output')
print('Output files:')
for p in sorted(out.glob('*.png')):
    print(p.name)
    display(DisplayImage(filename=str(p), width=700))


## Что исправлено для T4

На Tesla T4 SAM3 `addmm_act` внутри ViT MLP может вернуть `BFloat16` после `fc1`, хотя веса `fc2` остаются `Float32`. Поэтому простой `model.float()` недостаточен. В репозитории fallback переводит выход первой MLP-ветки обратно в `float32` **непосредственно перед `fc2`**, а inference запускается без CUDA autocast.

То есть ноутбук сохраняет рабочую архитектуру из твоей копии: отдельный Python 3.12 `.venv312`, тот же PyTorch/CUDA stack и запуск pipeline отдельным процессом.